# 11 · Red neuronal densa como challenger

**Objetivo.** Comprobar si una red neuronal densa pequeña mejora al HistGradientBoosting seleccionado sin cambiar features, folds ni ajustes posteriores.

**Arquitectura.** `MLPRegressor` con dos capas ocultas de 64 y 32 neuronas ReLU. La entrada y el objetivo se estandarizan con estadísticas del entrenamiento. Adam usa parada temprana sobre una fracción interna que contiene sólo elecciones anteriores al target.

**Control.** Se conservan `territorial_context`, folds 2018/2021/2024, pesos por registros identificados, baseline de misma familia, contracción residual 0.25 y corrección nacional.

**Gate.** Debe mejorar el MAE ponderado 2021 y no deteriorar 2018/2024 más de 0.5 pp.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display

from radar27.modeling.neural_challenger import (
    run_dense_neural_challenger,
    write_dense_neural_decision,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs/dense_neural_challenger.yaml"
        ).exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
with (PROJECT_ROOT / "configs/dense_neural_challenger.yaml").open(encoding="utf-8") as stream:
    neural_config = yaml.safe_load(stream)
mart_path = PROJECT_ROOT / "artifacts/tables/feature_ablation_mart.parquet"
reference_path = PROJECT_ROOT / "artifacts/tables/tabular_challenger_metrics.csv"
assert mart_path.exists() and reference_path.exists()
neural_mart = pd.read_parquet(mart_path)
display(pd.Series(neural_config["network"], name="valor").to_frame())

## Backtests fuera de tiempo

La red estima el mismo residual que los árboles. El objetivo se estandariza dentro de cada fold porque los residuales tienen una escala pequeña; la transformación se invierte antes de medir MAE.

In [ ]:
neural_predictions, neural_metrics, neural_registry = (
    run_dense_neural_challenger(PROJECT_ROOT, neural_mart)
)
metric_view = neural_metrics.copy()
for column in [
    "baseline_weighted_mae", "identified_weighted_mae",
    "reference_weighted_mae", "mae_change_vs_reference",
]:
    metric_view[column] = 100 * metric_view[column]
display(metric_view[[
    "target_year", "identified_weighted_mae", "reference_weighted_mae",
    "mae_change_vs_reference", "direct_iterations", "residual_iterations",
    "fit_seconds",
]])

In [ ]:
audit_checks = {
    "same_feature_group": set(neural_metrics.feature_group) == {"territorial_context"},
    "all_folds": set(neural_metrics.target_year) == {2018, 2021, 2024},
    "training_strictly_prior": all(
        all(int(year) < int(row.target_year) for year in str(row.training_target_years).split("|"))
        for row in neural_registry.itertuples(index=False)
    ),
    "positive_target_scales": bool(
        neural_registry.direct_target_scale.gt(0).all()
        and neural_registry.residual_target_scale.gt(0).all()
    ),
    "finite_predictions": bool(neural_predictions.prediction.notna().all()),
}
display(pd.Series(audit_checks, name="aprobado").to_frame())
assert all(audit_checks.values())

In [ ]:
neural_decision = write_dense_neural_decision(PROJECT_ROOT, neural_metrics)
display(pd.Series(neural_decision).to_frame("valor"))

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/dense_neural_challenger.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Red neuronal densa</h1>",
    "<p>Generado por notebooks/11_dense_neural_challenger.ipynb.</p>",
    "<h2>Métricas</h2>", neural_metrics.to_html(index=False),
    "<h2>Registro</h2>", neural_registry.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(
        neural_decision, ensure_ascii=False, indent=2
    ) + "</pre>",
]), encoding="utf-8")
required_outputs = [
    PROJECT_ROOT / "artifacts/tables/dense_neural_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/dense_neural_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/dense_neural_registry.csv",
    PROJECT_ROOT / "artifacts/dense_neural_decision.json",
    report_path,
]
final_checks = {"all_outputs_exist": all(path.exists() for path in required_outputs), **audit_checks}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
display(Markdown(
    f"**Cierre.** MAE 2021 de la red: "
    f"`{100*neural_decision['challenger_primary_weighted_mae']:.2f} pp`; "
    f"referencia: `{100*neural_decision['reference_primary_weighted_mae']:.2f} pp`; "
    f"promovida: `{neural_decision['challenger_promoted']}`. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))